In [0]:
df = spark.table("bronze_layer.crashes_table")

In [0]:
df.count()

In [0]:
from pyspark.sql.functions import col, to_timestamp

df = df \
    .withColumn("crash_record_id", col("crash_record_id").cast("string")) \
    .withColumn("crash_date", to_timestamp("crash_date")) \
    .withColumn("posted_speed_limit", col("posted_speed_limit").cast("int")) \
    .withColumn("traffic_control_device", col("traffic_control_device").cast("string")) \
    .withColumn("device_condition", col("device_condition").cast("string")) \
    .withColumn("weather_condition", col("weather_condition").cast("string")) \
    .withColumn("lighting_condition", col("lighting_condition").cast("string")) \
    .withColumn("first_crash_type", col("first_crash_type").cast("string")) \
    .withColumn("trafficway_type", col("trafficway_type").cast("string")) \
    .withColumn("lane_cnt", col("lane_cnt").cast("string")) \
    .withColumn("alignment", col("alignment").cast("string")) \
    .withColumn("roadway_surface_cond", col("roadway_surface_cond").cast("string")) \
    .withColumn("road_defect", col("road_defect").cast("string")) \
    .withColumn("report_type", col("report_type").cast("string")) \
    .withColumn("crash_type", col("crash_type").cast("string")) \
    .withColumn("intersection_related_i", col("intersection_related_i").cast("string")) \
    .withColumn("hit_and_run_i", col("hit_and_run_i").cast("string")) \
    .withColumn("damage", col("damage").cast("string")) \
    .withColumn("date_police_notified", to_timestamp("date_police_notified")) \
    .withColumn("prim_contributory_cause", col("prim_contributory_cause").cast("string")) \
    .withColumn("sec_contributory_cause", col("sec_contributory_cause").cast("string")) \
    .withColumn("street_no", col("street_no").cast("int")) \
    .withColumn("street_direction", col("street_direction").cast("string")) \
    .withColumn("street_name", col("street_name").cast("string")) \
    .withColumn("beat_of_occurrence", col("beat_of_occurrence").cast("string")) \
    .withColumn("num_units", col("num_units").cast("int")) \
    .withColumn("most_severe_injury", col("most_severe_injury").cast("string")) \
    .withColumn("injuries_total", col("injuries_total").cast("int")) \
    .withColumn("injuries_fatal", col("injuries_fatal").cast("int")) \
    .withColumn("injuries_incapacitating", col("injuries_incapacitating").cast("int")) \
    .withColumn("injuries_non_incapacitating", col("injuries_non_incapacitating").cast("int")) \
    .withColumn("injuries_reported_not_evident", col("injuries_reported_not_evident").cast("int")) \
    .withColumn("injuries_no_indication", col("injuries_no_indication").cast("int")) \
    .withColumn("injuries_unknown", col("injuries_unknown").cast("int")) \
    .withColumn("crash_hour", col("crash_hour").cast("int")) \
    .withColumn("crash_day_of_week", col("crash_day_of_week").cast("int")) \
    .withColumn("crash_month", col("crash_month").cast("int")) \
    .withColumn("latitude", col("latitude").cast("double")) \
    .withColumn("longitude", col("longitude").cast("double"))

In [0]:
from pyspark.sql.types import * 
from pyspark.sql.functions import * 
# df = df.withColumn("CRASH_DATE", substring_index("CRASH_DATE", " ", 1))
# df = df.withColumn("CRASH_DATE", to_date(col("CRASH_DATE"), "MM/dd/yyyy"))

for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name,trim(col(field.name)))

In [0]:
from pyspark.sql.functions import sum 
cols_to_keep = []
df_nulls = df.select([
    sum(col(c).isNull().cast('int')).alias(c)
    for c in df.columns
]).collect()[0].asDict()


for key, value in df_nulls.items():
    if value < 10000:
        cols_to_keep.append(key)
        


df = df.select(cols_to_keep)

In [0]:
string_nulls = ["", " ", "NA", "N/A", "null", "None"]
numeric_nulls = [0, -1, 9999]
df_clean =df

df.dtypes

for column, data_type in df.dtypes:
    if data_type == 'string':
        df_clean = df.withColumn(column, when(col(column).isNull() | (trim(col(column)).isin(string_nulls)),None) \
                                 .otherwise(col(column)))
        
    elif data_type in ("int", "bigint", "double", "float"):
        df_clean = df.withColumn(column, when(col(column).isNull() | (col(column).isin(numeric_nulls)),0) \
                                 .otherwise(col(column)))
         
        
df_clean = df_clean.fillna({"latitude": 0})
df_clean = df_clean.fillna({"longitude": 0})
injuries_cols = ["INJURIES_TOTAL", "INJURIES_FATAL", "INJURIES_INCAPACITATING", "INJURIES_NON_INCAPACITATING", "INJURIES_REPORTED_NOT_EVIDENT", "INJURIES_NO_INDICATION", "INJURIES_UNKNOWN"]

for x in injuries_cols:
    df_clean = df_clean.fillna({f"{x}": 0})

In [0]:
df_clean.createOrReplaceTempView("crashes_silver_table")

In [0]:
final_crash_df =spark.sql( """
          SELECT
          *,
            CASE 
            WHEN UPPER(DAMAGE) LIKE '%OVER%1%500%' THEN 2000
            WHEN UPPER(DAMAGE) LIKE '%500%OR%LESS' THEN 500
            WHEN UPPER(DAMAGE) LIKE '%501%' THEN 1500
            ELSE 0
            END AS damage_cost_estimate
          
          FROM crashes_silver_table
          
          
          
          """)


In [0]:
%sql
CREATE DATABASE IF NOT EXISTS silver_layer;

In [0]:
final_crash_df.write \
    .mode('overwrite') \
    .format("delta") \
    .save("abfss://source@datalakeimewore.dfs.core.windows.net/ChicagoCrashes/silver/crashes")

In [0]:
%sql
CREATE TABLE silver_layer.crashes
USING DELTA
LOCATION 'abfss://source@datalakeimewore.dfs.core.windows.net/ChicagoCrashes/silver/crashes';